![MuJoCo banner](https://raw.githubusercontent.com/google-deepmind/mujoco/main/banner.png)

# <h1><center>Electrical Motor Visualizer on Colab <a href="https://colab.research.google.com/github/robomotic/mujoco/blob/motors/python/examples/electrical/demo_visualizer_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" width="140" align="center"/></a></center></h1>

This notebook will:

1. fetch the remote `motors` branch,
2. install the released MuJoCo wheel from GitHub Releases,
3. run the electrical visualizer demo, and
4. expose a browser port so the viewer can be opened inside Google Colab.

> The notebook prefers released wheels and now supports Python `3.11` and `3.12` runtimes, selecting the matching wheel automatically. If the requested wheel is not yet published, it falls back to building MuJoCo from source in Colab.

In [1]:
from pathlib import Path
import os
import platform
import sys

REPO_URL = 'https://github.com/robomotic/mujoco.git'
BRANCH = 'motors'
RELEASE_TAG = 'motors-wheel-v3.7.0-8'
REPO_DIR = Path('/content/mujoco')
VNC_PORT = 6080
DISPLAY_ID = ':1'

SUPPORTED_WHEELS = {
    (3, 11): 'mujoco-3.7.0-cp311-cp311-linux_x86_64.whl',
    (3, 12): 'mujoco-3.7.0-cp312-cp312-linux_x86_64.whl',
}

py_key = sys.version_info[:2]
if py_key not in SUPPORTED_WHEELS:
    raise RuntimeError(
        f'Unsupported Colab runtime Python {sys.version.split()[0]}. '
        'Please switch to Python 3.11 or 3.12 (Runtime → Change runtime type).'
    )
WHEEL_NAME = SUPPORTED_WHEELS[py_key]
WHEEL_URL = f'https://github.com/robomotic/mujoco/releases/download/{RELEASE_TAG}/{WHEEL_NAME}'

os.environ['REPO_URL'] = REPO_URL
os.environ['BRANCH'] = BRANCH
os.environ['REPO_DIR'] = str(REPO_DIR)
os.environ['WHEEL_URL'] = WHEEL_URL
os.environ['VNC_PORT'] = str(VNC_PORT)
os.environ['DISPLAY_ID'] = DISPLAY_ID

print(f'Python:   {sys.version.split()[0]}')
print(f'Platform: {platform.platform()}')
print(f'Repo:     {REPO_URL}')
print(f'Branch:   {BRANCH}')
print(f'Release:  {RELEASE_TAG}')
print(f'Wheel:    {WHEEL_URL}')
print(f'Port:     {VNC_PORT}')


Python:   3.12.13
Platform: Linux-6.6.113+-x86_64-with-glibc2.35
Repo:     https://github.com/robomotic/mujoco.git
Branch:   motors
Release:  motors-wheel-v3.7.0-6
Wheel:    https://github.com/robomotic/mujoco/releases/download/motors-wheel-v3.7.0-6/mujoco-3.7.0-cp312-cp312-linux_x86_64.whl
Port:     6080


In [2]:
%%bash
set -euxo pipefail
apt-get update
DEBIAN_FRONTEND=noninteractive apt-get install -y \
  git libgl1-mesa-glx libglfw3 libosmesa6 mesa-utils \
  xvfb fluxbox x11vnc websockify novnc \
  build-essential cmake ninja-build python3-dev python3-venv pkg-config \
  libgl1-mesa-dev libwayland-dev libxinerama-dev libxcursor-dev libxkbcommon-dev \
  libxrandr-dev libxi-dev
python3 -m pip install --upgrade pip
python3 -m pip install --upgrade \
  "$WHEEL_URL" \
  glfw PyOpenGL absl-py "etils[epath]" || {
  echo "Wheel install failed; falling back to source build."
  pip install --upgrade pip setuptools wheel build
  if [ ! -d "$REPO_DIR/.git" ]; then
    git clone --depth=1 --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
  fi
  cd "$REPO_DIR/python"
  python3 ./make_sdist.sh
  cd dist
  pip wheel --no-deps mujoco-*.tar.gz
  pip install --no-index mujoco-*.whl
}
python3 - <<'PY'
import mujoco
print('Installed MuJoCo version:', mujoco.__version__)
print('Bundled plugin dir:', mujoco.PLUGINS_DIR)
PY

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [89.0 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [356 B]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,988 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,292 kB]
Get:14 https

+ apt-get update
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
+ DEBIAN_FRONTEND=noninteractive
+ apt-get install -y git libgl1-mesa-glx libglfw3 libosmesa6 mesa-utils xvfb fluxbox x11vnc websockify novnc build-essential cmake ninja-build python3-dev python3-venv pkg-config libgl1-mesa-dev libwayland-dev libxinerama-dev libxcursor-dev libxkbcommon-dev libxrandr-dev libxi-dev
+ python3 -m pip install --upgrade pip
+ python3 -m pip install --upgrade https://github.com/robomotic/mujoco/releases/download/motors-wheel-v3.7.0-6/mujoco-3.7.0-cp312-cp312-linux_x86_64.whl glfw PyOpenGL absl-py 'etils[epath]'
  ERROR: HTTP error 404 while getting https://github.com/robomotic/mujoco/releases/download/motors-wheel-v3.7.0-6/mujoco-3.7.0-cp312-cp312-linux_x86_64.whl
ERROR: Could not install requirement mujoco==3.7.0 from https://github.com/robomotic/mujoco/re

CalledProcessError: Command 'b'set -euxo pipefail\napt-get update\nDEBIAN_FRONTEND=noninteractive apt-get install -y \\\n  git libgl1-mesa-glx libglfw3 libosmesa6 mesa-utils \\\n  xvfb fluxbox x11vnc websockify novnc \\\n  build-essential cmake ninja-build python3-dev python3-venv pkg-config \\\n  libgl1-mesa-dev libwayland-dev libxinerama-dev libxcursor-dev libxkbcommon-dev \\\n  libxrandr-dev libxi-dev\npython3 -m pip install --upgrade pip\npython3 -m pip install --upgrade \\\n  "$WHEEL_URL" \\\n  glfw PyOpenGL absl-py "etils[epath]" || {\n  echo "Wheel install failed; falling back to source build."\n  python3 -m venv /tmp/mujoco-venv\n  source /tmp/mujoco-venv/bin/activate\n  python -m pip install --upgrade pip setuptools wheel build\n  if [ ! -d "$REPO_DIR/.git" ]; then\n    git clone --depth=1 --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"\n  fi\n  cd "$REPO_DIR/python"\n  python ./make_sdist.sh\n  cd dist\n  python -m pip wheel --no-deps mujoco-*.tar.gz\n  python -m pip install --no-

In [ ]:
%%bash
set -euxo pipefail
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone --depth=1 --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
fi
cd "$REPO_DIR"
git fetch origin "$BRANCH" --depth=1
git checkout "$BRANCH"
git pull --ff-only origin "$BRANCH"
git status --short --branch


In [ ]:
import mujoco
import mujoco.viewer
from mujoco.electrical import SingleEnvSimulation

print('Wheel import OK:', mujoco.__version__)
print('Bundled plugin dir:', mujoco.PLUGINS_DIR)
print('Demo path:', REPO_DIR / 'python/examples/electrical/demo_visualizer.py')
print('Electrical sim class:', SingleEnvSimulation)


In [ ]:
import os
import subprocess
import time
from google.colab import output

os.environ['DISPLAY'] = DISPLAY_ID
os.environ['LIBGL_ALWAYS_SOFTWARE'] = '1'

_bg_processes = globals().get('_bg_processes', {})

def start_once(name, cmd, env=None):
    proc = _bg_processes.get(name)
    if proc is not None and proc.poll() is None:
        print(f'{name} already running (pid={proc.pid})')
        return proc
    proc = subprocess.Popen(cmd, env=env or os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)
    _bg_processes[name] = proc
    print(f'started {name} (pid={proc.pid})')
    return proc

start_once('xvfb', ['Xvfb', DISPLAY_ID, '-screen', '0', '1440x900x24', '-ac', '+extension', 'GLX', '+render'])
time.sleep(2)
start_once('fluxbox', ['fluxbox'], env={**os.environ, 'DISPLAY': DISPLAY_ID})
start_once('x11vnc', ['x11vnc', '-display', DISPLAY_ID, '-forever', '-shared', '-nopw', '-rfbport', '5901'])
start_once('novnc', ['websockify', '--web=/usr/share/novnc/', str(VNC_PORT), 'localhost:5901'])

print(f'Opening noVNC on port {VNC_PORT}...')
output.serve_kernel_port_as_iframe(VNC_PORT, path='/vnc.html?autoconnect=true&resize=scale', height=720)

In [ ]:
import os
import subprocess

env = os.environ.copy()
env['DISPLAY'] = DISPLAY_ID
env['LIBGL_ALWAYS_SOFTWARE'] = '1'

demo_cmd = [
    'python3',
    str(REPO_DIR / 'python/examples/electrical/demo_visualizer.py'),
    '--light',
    '--steps',
    '5000',
]

demo_proc = subprocess.Popen(demo_cmd, cwd=str(REPO_DIR), env=env)
print(f'Visualizer started with PID {demo_proc.pid}.')
print('Open the embedded noVNC pane above, then press F4 inside the viewer for the sensor panel.')


## Optional helpers

- The setup cell automatically selects the `cp311` or `cp312` wheel based on the active Colab runtime.
- To use a newer published wheel later, update `RELEASE_TAG` in the second cell.
- Re-run the **noVNC** cell if the browser frame disconnects.
- Change `--light` to `--heavy` in the launch cell to run the heavier payload scenario.
- If you want to stop the current viewer, run:

```python
import os, signal
os.kill(demo_proc.pid, signal.SIGTERM)
```
